# Building Capacity Analysis

This notebook provides a comprehensive analysis of building capacities based on geographical factors.

In [8]:
import pandas as pd

from analysis.building_levels.building_analysis import CapacityAnalyzer, get_path

analyzer = CapacityAnalyzer()
print("Analyzer initialized.")

Loading live game data...
Analyzer initialized.


## Modifier Sources Breakdown

In [9]:
df_sources = analyzer.get_modifier_sources_df()
display(df_sources)

,Category,Source,Fruit Orchard,Sheep Farm,Farming Village,Fishing Village,Forest Village,Row Sum
0,Climate,mediterranean,8.0,4.0,5.0,4.0,0.0,21.0
1,Climate,subtropical,4.0,0.0,5.0,0.0,0.0,9.0
2,Climate,arid,1.0,6.0,0.0,0.0,0.0,7.0
3,Climate,oceanic,2.0,4.0,3.0,7.0,0.0,16.0
4,Climate,continental,1.0,3.0,6.0,2.0,5.0,17.0
5,Climate,arctic,0.0,0.0,0.0,4.0,8.0,12.0
6,Climate,tropical,0.0,0.0,2.0,3.0,2.0,7.0
7,Climate,cold_arid,0.0,5.0,0.0,0.0,0.0,5.0
8,Climate,TOTAL CLIMATE,16.0,22.0,21.0,20.0,15.0,94.0
9,,,NaN,NaN,NaN,NaN,NaN,NaN


## Generalized Analysis Interface

The `CapacityAnalyzer` now supports generalized filtering and analysis methods. This allows for quick custom queries without writing new scripts.

In [10]:
df_locations = analyzer.location_data.get_merged_df()
print("Locations loaded.")

Locations loaded.


### Query by Country TAG

Single aggregate sheet: population, development, building capacity (and relative share), employment and employment/pop ratio per building, geography breakdown (climate/topography/vegetation %).

In [14]:
TAG = "ENG"  # Change to query any country (e.g. FRA, ENG, POL)

country_locs = df_locations[df_locations["owner_tag"] == TAG].copy()
country_locs = country_locs.dropna(subset=["climate", "topography", "vegetation"])
if "rank" not in country_locs.columns:
    country_locs["rank"] = "rural_settlement"

n_loc = len(country_locs)
if n_loc == 0:
    print("No owned locations found. Check TAG.")
else:
    total_pop = country_locs["population"].sum()
    avg_dev = country_locs["development"].mean().round(2)
    df_cap = analyzer.calculate_capacities_for_locations(country_locs)
    building_cols = ["Fruit Orchard", "Sheep Farm", "Farming Village", "Fishing Village", "Forest Village"]
    cap_sum = df_cap.groupby("Building")["Total Bonus"].sum().reindex(building_cols)
    total_cap = cap_sum.sum()
    bd = analyzer.building_data.modded_df
    building_keys = {"Fruit Orchard": "fruit_orchard", "Sheep Farm": "sheep_farms", "Farming Village": "farming_village", "Fishing Village": "fishing_village", "Forest Village": "forest_village"}
    emp_vals = [bd.loc[building_keys[b], "employment_size_val"] if building_keys[b] in bd.index else 0 for b in building_cols]
    employment = cap_sum * pd.Series(emp_vals, index=building_cols)
    emp_ratio = (employment / total_pop * 100).round(2) if total_pop > 0 else pd.Series(float("nan"), index=building_cols)

    # Build single aggregate row
    row = {
        "TAG": TAG,
        "n_locations": n_loc,
        "total_population_k": round(total_pop, 1),
        "avg_development": avg_dev,
    }
    for b in building_cols:
        c, e, r = cap_sum.get(b, 0), employment.get(b, 0), emp_ratio.get(b, float("nan"))
        pct = (c / total_cap * 100) if total_cap > 0 else 0
        short = b.replace(" ", "_").lower()
        row[f"{short}_cap"] = round(c, 1)
        row[f"{short}_pct"] = round(pct, 1)
        row[f"{short}_emp_k"] = round(e, 1)
        row[f"{short}_emp_pop_pct"] = r if pd.notna(r) else "-"
    for col in ["climate", "topography", "vegetation"]:
        if col not in country_locs.columns:
            continue
        vc = country_locs[col].value_counts(normalize=True) * 100
        for val, pct in vc.items():
            row[f"{col}_{val}_pct"] = round(pct, 1)

    # Order columns: core, buildings (cap/pct/emp/emp_pop), geography
    core = ["TAG", "n_locations", "total_population_k", "avg_development"]
    building_cols_short = ["fruit_orchard", "sheep_farm", "farming_village", "fishing_village", "forest_village"]
    building_cols_ordered = [f"{b}_{s}" for b in building_cols_short for s in ["cap", "pct", "emp_k", "emp_pop_pct"]]
    geo_cols = [c for c in row if c not in core and c not in building_cols_ordered]
    df_country = pd.DataFrame([row])[[c for c in core + building_cols_ordered + geo_cols if c in row]]
    display(df_country)

    # Compact view: fewer columns, stacked values, padded with empty where lengths differ
    summary_list = [TAG, f"{n_loc} loc", f"{total_pop:.0f}k pop", f"avg dev {avg_dev}"]
    veg_list = [f"{v}: {pct:.1f}%" for v, pct in country_locs["vegetation"].value_counts(normalize=True).mul(100).items()]
    topo_list = [f"{v}: {pct:.1f}%" for v, pct in country_locs["topography"].value_counts(normalize=True).mul(100).items()]
    climate_list = [f"{v}: {pct:.1f}%" for v, pct in country_locs["climate"].value_counts(normalize=True).mul(100).items()]
    cap_list = [f"{b}: {cap_sum.get(b, 0):.1f}" for b in building_cols]
    pct_list = [f"{b}: {(cap_sum.get(b,0)/total_cap*100):.1f}%" if total_cap > 0 else f"{b}: -" for b in building_cols]
    emp_list = [f"{b}: {employment.get(b, 0):.1f}k" for b in building_cols]
    emp_pop_list = [f"{b}: {emp_ratio.get(b):.1f}%" if pd.notna(emp_ratio.get(b)) else f"{b}: -" for b in building_cols]
    n_rows = max(1, len(veg_list), len(topo_list), len(climate_list), len(cap_list))
    def pad(lst, n):
        return lst + [""] * (n - len(lst))
    df_compact = pd.DataFrame({
        "Summary": pad(summary_list, n_rows),
        "Vegetation": pad(veg_list, n_rows),
        "Topography": pad(topo_list, n_rows),
        "Climate": pad(climate_list, n_rows),
        "Capacity": pad(cap_list, n_rows),
        "Relative Cap %": pad(pct_list, n_rows),
        "Employment": pad(emp_list, n_rows),
        "Emp/Pop %": pad(emp_pop_list, n_rows),
    })
    print("\nCompact view:")
    display(df_compact)

,TAG,n_locations,total_population_k,avg_development,fruit_orchard_cap,fruit_orchard_pct,fruit_orchard_emp_k,fruit_orchard_emp_pop_pct,sheep_farm_cap,sheep_farm_pct,...,forest_village_emp_pop_pct,climate_oceanic_pct,topography_flatland_pct,topography_wetlands_pct,topography_hills_pct,vegetation_grasslands_pct,vegetation_woods_pct,vegetation_sparse_pct,vegetation_farmland_pct,vegetation_forest_pct
0,ENG,138,3015.5,20.93,2265.5,19.4,2265.5,75.13,2105.1,18.0,...,59.91,100.0,87.7,6.5,5.8,42.8,22.5,12.3,11.6,10.9



Compact view:


,Summary,Vegetation,Topography,Climate,Capacity,Relative Cap %,Employment,Emp/Pop %
0,ENG,grasslands: 42.8%,flatland: 87.7%,oceanic: 100.0%,Fruit Orchard: 2265.5,Fruit Orchard: 19.4%,Fruit Orchard: 2265.5k,Fruit Orchard: 75.1%
1,138 loc,woods: 22.5%,wetlands: 6.5%,,Sheep Farm: 2105.1,Sheep Farm: 18.0%,Sheep Farm: 2105.1k,Sheep Farm: 69.8%
2,3016k pop,sparse: 12.3%,hills: 5.8%,,Farming Village: 2991.5,Farming Village: 25.6%,Farming Village: 2991.5k,Farming Village: 99.2%
3,avg dev 20.93,farmland: 11.6%,,,Fishing Village: 2533.5,Fishing Village: 21.7%,Fishing Village: 2533.5k,Fishing Village: 84.0%
4,,forest: 10.9%,,,Forest Village: 1806.5,Forest Village: 15.4%,Forest Village: 1806.5k,Forest Village: 59.9%


### Example: Top Regions in Europe

Using `run_standard_analysis` with a filter for `super_region`.

In [12]:
europe_results = analyzer.run_standard_analysis(df_locations, filters={'super_region': 'europe'})

for building, data in europe_results.items():
    print(f"\nTop European Regions for {building}:")
    display(data[['Total Bonus_mean', 'Location Count']])


Top European Regions for Fruit Orchard:


,Total Bonus_mean,Location Count
Region,,
italy_region,18.95,359
france_region,16.67,481
iberia_region,16.28,497
north_german_region,15.85,419
balkan_region,14.95,471
south_german_region,14.46,380
great_britain_region,14.15,304
carpathia_region,11.86,287
baltic_region,11.73,364



Top European Regions for Sheep Farm:


,Total Bonus_mean,Location Count
Region,,
iberia_region,15.29,497
france_region,14.59,481
north_german_region,14.49,419
italy_region,14.19,359
great_britain_region,14.18,304
south_german_region,13.95,380
steppes_region,12.61,327
caucasus_region,12.01,221
balkan_region,11.99,471



Top European Regions for Farming Village:


,Total Bonus_mean,Location Count
Region,,
north_german_region,21.64,419
france_region,20.61,481
italy_region,20.39,359
baltic_region,20.38,364
south_german_region,20.33,380
ruthenia_region,20.10,306
carpathia_region,17.87,287
great_britain_region,17.66,304
iberia_region,17.13,497



Top European Regions for Fishing Village:


,Total Bonus_mean,Location Count
Region,,
great_britain_region,16.96,304
france_region,16.45,481
north_german_region,14.96,419
ireland_region,14.47,135
italy_region,14.35,359
iberia_region,13.76,497
south_german_region,11.83,380
balkan_region,11.23,471
baltic_region,9.67,364



Top European Regions for Forest Village:


,Total Bonus_mean,Location Count
Region,,
ural_region,18.99,211
south_german_region,18.80,380
ruthenia_region,18.43,306
russian_region,18.28,594
baltic_region,17.55,364
carpathia_region,17.04,287
scandinavian_region,15.89,629
north_german_region,14.91,419
italy_region,14.51,359


### Example: Outlier Analysis for Scandinavia

Using `get_outlier_analysis` with a filter for `region`.

In [13]:
top, bottom = analyzer.get_outlier_analysis(df_locations, building_name="Fruit Orchard", filters={'region': 'scandinavian_region'})

print("Top Outliers in Scandinavia for Fruit Orchard:")
display(top[['Location', 'Province', 'Total Bonus', 'Climate', 'Topography', 'Vegetation']])

Top Outliers in Scandinavia for Fruit Orchard:


,Location,Province,Total Bonus,Climate,Topography,Vegetation
118,odense,funen_province,19.19,oceanic,flatland,farmland
332,bergen,bergenhus_province,18.90,oceanic,hills,grasslands
96,aalborg,norrejylland_province,18.02,oceanic,flatland,grasslands
110,roskilde,zealand_province,17.92,continental,flatland,farmland
103,aarhus,eastern_jutland_province,17.91,continental,flatland,farmland
115,kobenhavn,zealand_province,17.86,continental,flatland,farmland
124,flensburg,slesvig_province,17.86,oceanic,flatland,grasslands
130,malmo,malmohus_province,17.67,continental,flatland,farmland
385,molde,romsdalen_province,17.63,oceanic,hills,grasslands
113,helsingor,zealand_province,17.13,continental,flatland,grasslands
